# 00 · Collect raw facade images from Wikimedia Commons

**Purpose:** gather candidate photos per facade-material class, with a license and attribution record for every image. The output is uploaded to Roboflow for annotation. This notebook is **not** part of the training pipeline; it documents where the raw images came from.

**Why Wikimedia Commons only:** every file on Commons carries a free license that allows redistribution with attribution. Unsplash / Pexels licenses generally forbid repackaging their photos into a redistributed dataset, which is exactly what the GitHub Release zip is.

**Run it in one sitting.** Colab's `/content/` is wiped when the session resets, so run cell 1 → review → run cell 3 without leaving the session idle.

Outputs:
- `/content/raw_images/<class>/<class>_NNN.jpg|png`
- `/content/raw_images/sources.csv`: one row per image (Commons page, license, license URL, artist). This is the attribution record for the dataset's license section.

In [ ]:
import csv, html, re, time
from pathlib import Path
import requests

API = "https://commons.wikimedia.org/w/api.php"
# Wikimedia requires a descriptive User-Agent; generic clients can be blocked.
HEADERS = {"User-Agent": "FacadeMaterialsDataset/1.0 (https://github.com/OmarEAbdelaal/M4U3-CV-Assignment-Facade-Materials)"}
OUT_ROOT = Path("/content/raw_images")
PER_CLASS = 50
MIN_WIDTH = 640

# Keys = class names (must match the Roboflow class names exactly). Values = search phrases.
CLASSES = {
    "glass": "glass facade building",
    "painted_render": "painted stucco wall building",
    "exposed_concrete": "exposed concrete facade architecture",
    "brick": "brick wall building facade",
    "cladding_panel": "metal cladding facade building",
    "stone_cladding": "stone cladding facade building",
}

# Public domain, CC0, CC BY and CC BY-SA (any version). Everything else (e.g. GFDL-only) is skipped.
ALLOWED_LICENSE = re.compile(r"^(public domain|cc0|cc by(-sa)? \d)", re.I)
EXTENSIONS = {"image/jpeg": ".jpg", "image/png": ".png", "image/webp": ".webp"}


def plain(value):
    return html.unescape(re.sub(r"<[^>]+>", "", value or "")).strip()


def search_pages(session, query):
    params = {
        "action": "query", "format": "json",
        "generator": "search", "gsrsearch": f"{query} filetype:bitmap",
        "gsrnamespace": 6, "gsrlimit": 50,
        "prop": "imageinfo", "iiprop": "url|size|extmetadata", "iiurlwidth": 1024,
    }
    while True:
        resp = session.get(API, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        pages = data.get("query", {}).get("pages", {}).values()
        yield from sorted(pages, key=lambda p: p.get("index", 0))
        if "continue" not in data:
            return
        params.update(data["continue"])


def collect(session, cls, query, seen, rows):
    out_dir = OUT_ROOT / cls
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = skipped_license = 0
    for page in search_pages(session, query):
        if saved >= PER_CLASS:
            break
        if page["pageid"] in seen or "imageinfo" not in page:
            continue
        info = page["imageinfo"][0]
        meta = info.get("extmetadata", {})
        license_name = plain(meta.get("LicenseShortName", {}).get("value"))
        if not ALLOWED_LICENSE.match(license_name):
            skipped_license += 1
            continue
        if info.get("width", 0) < MIN_WIDTH:
            continue
        try:
            resp = session.get(info.get("thumburl") or info["url"], timeout=30)
        except requests.RequestException as e:
            print(f"  skip {page['title']}: {e}")
            continue
        ext = EXTENSIONS.get(resp.headers.get("Content-Type", "").split(";")[0])
        if not resp.ok or ext is None:
            print(f"  skip {page['title']}: HTTP {resp.status_code} {resp.headers.get('Content-Type')}")
            continue
        seen.add(page["pageid"])
        fname = f"{cls}_{saved:03d}{ext}"
        (out_dir / fname).write_bytes(resp.content)
        rows.append({
            "file": f"{cls}/{fname}",
            "class": cls,
            "commons_page": info.get("descriptionurl", ""),
            "license": license_name,
            "license_url": plain(meta.get("LicenseUrl", {}).get("value")),
            "artist": plain(meta.get("Artist", {}).get("value")) or "unknown",
        })
        saved += 1
        time.sleep(0.5)
    print(f"{cls}: saved {saved} (skipped {skipped_license} for license)")


session = requests.Session()
session.headers.update(HEADERS)
seen, rows = set(), []  # `seen` stops one Commons file landing in two classes (train/val leakage)
for cls, query in CLASSES.items():
    collect(session, cls, query, seen, rows)

with open(OUT_ROOT / "sources.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file", "class", "commons_page", "license", "license_url", "artist"])
    writer.writeheader()
    writer.writerows(rows)
print(f"\nTotal: {len(rows)} images. Attribution record: {OUT_ROOT / 'sources.csv'}")

glass: saved 50 (skipped 0 for license)


## Review before exporting

Keyword search is not precise. Open the folder icon in Colab's left sidebar → `raw_images` → each class folder, preview images, and **delete** (right-click → Delete) anything that is:

- not a building facade (interiors, close-up textures with no building context, maps, drawings)
- dominated by a material other than the folder's class
- a near-duplicate of another image you're keeping

If a class came back thin or noisy, edit its search phrase in cell 1 and re-run.

Deleting files is enough; the next cell drops their rows from `sources.csv` automatically.

In [ ]:
import csv, shutil
from pathlib import Path
from google.colab import files

OUT_ROOT = Path("/content/raw_images")
csv_path = OUT_ROOT / "sources.csv"

with open(csv_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    fieldnames, rows = reader.fieldnames, list(reader)
kept = [r for r in rows if (OUT_ROOT / r["file"]).exists()]
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(kept)

print(f"Kept {len(kept)} of {len(rows)} images after review")
for cls in sorted({r["class"] for r in kept}):
    print(f"  {cls}: {sum(r['class'] == cls for r in kept)}")

shutil.make_archive("/content/raw_images_all", "zip", OUT_ROOT)
files.download("/content/raw_images_all.zip")

## Next: upload to Roboflow

1. Unzip `raw_images_all.zip` on your computer.
2. In your Roboflow project → **Upload**, drag in one class folder at a time and use the folder name as the batch name. That keeps provenance traceable.
3. Folder names do **not** create labels. This is object detection (mAP50), so every image still needs bounding boxes drawn in Roboflow, following the label rules in `docs/02_class_definitions.md`.
4. Keep `sources.csv`. It goes into the repo as `data/sources.csv`, the attribution record the dataset license depends on.